# Masiez — Train RT-DETR Target Detector (Kaggle) → ONNX for the Browser

Train an **RT-DETR** detector that finds hostile **tanks** and **soldiers** in the drone's FPV camera image, then export it to **ONNX** so the game can run inference fully **in-browser** (ONNX Runtime Web) with no external AI server.

## How to run on Kaggle
1. Open a new Kaggle Notebook, attach GPU accelerator.
2. *(Optional)* attach a Masiez capture dataset (frames + YOLO-format labels). If none is attached, the notebook generates a synthetic drone-view dataset so it runs end-to-end out of the box.
3. Run all cells, then download the exported `rt-detr.onnx` + `rt-detr.labels.txt` from the notebook output and drop them into `game/public/models/` in the Masiez repo.

## Browser interface (must stay fixed)
- Input: float RGB `1x3x640x640`, values in **0..1**, letterboxed onto gray.
- Output: Ultralytics RT-DETR `output0` of shape **`1x300x6`** — x1, y1, x2, y2, confidence, class-id (boxes in 640-space pixels).

In [ ]:
%pip install -q ultralytics opencv-python-headless onnx onnxruntime

## 1. Configuration

In [ ]:
import os, random, shutil
import numpy as np
import cv2

IMG_SIZE = 640
CLASSES = ['tank', 'soldier']          # class index == label id (keep in sync with rt-detr.labels.txt)
WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else './rtdetr_work'
DATASET_ROOT = os.path.join(WORK, 'rtdetr_dataset')

def find_attached_dataset():
    for p in os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else []:
        cand = os.path.join('/kaggle/input', p)
        if os.path.isdir(os.path.join(cand, 'images')) or os.path.isdir(os.path.join(cand, 'train')):
            return cand
    return None

attached = find_attached_dataset()
print('Attached dataset:', attached or 'none — using the synthetic generator below')

## 2. Dataset

**Option A** — an attached Masiez capture dataset in YOLO layout (`images/train`, `labels/train`, ...) is used as-is (set `USE_ATTACHED = True`).
**Option B** — synthetic drone-view frames: a blurred sky/ground backdrop with terrain clutter, boxy tanks and small soldiers placed at random depth (perspective-scaled), written with YOLO labels.

In [ ]:
USE_ATTACHED = attached is not None
LABELS = os.path.join(DATASET_ROOT, 'labels')
IMAGES = os.path.join(DATASET_ROOT, 'images')

def rand_bg(size, rng):
    img = np.zeros((size, size, 3), np.uint8)
    horizon = rng.randint(int(size * 0.30), int(size * 0.55))
    sky = (rng.randint(140, 190), rng.randint(170, 210), rng.randint(190, 235))   # BGR
    ground = np.array([rng.randint(40, 90), rng.randint(70, 130), rng.randint(60, 110)], np.int16)
    img[:horizon] = sky
    for y in range(horizon, size):
        t = (y - horizon) / (size - horizon)
        img[y] = np.clip(ground * (1.0 + 1.4 * t), 0, 255).astype(np.uint8)
    for _ in range(rng.randint(10, 30)):   # terrain clutter patches
        cx, cy = rng.randrange(size), rng.randint(horizon, size - 1)
        s = rng.randint(3, 22)
        col = tuple(int(c) for c in np.clip(np.array(img[cy, cx], np.int16) + rng.randint(-30, 30), 0, 255))
        cv2.circle(img, (cx, cy), s, col, -1)
    return cv2.GaussianBlur(img, (5, 5), 0)

def draw_target(img, cx, cy, w, h, cls, rng):
    if cls == 0:  # tank: wide dark hull, turret and barrel
        cv2.rectangle(img, (cx - w // 2, cy - h // 2), (cx + w // 2, cy + h // 2),
                      (rng.randint(25, 50), rng.randint(30, 55), rng.randint(30, 55)), -1)
        cv2.rectangle(img, (cx - w // 6, cy - h), (cx + w // 6, cy - h // 2),
                      (40, 45, 45), -1)
        cv2.line(img, (cx, cy - 3 * h // 4), (cx + w // 2, cy - 3 * h // 4), (35, 40, 40), max(1, h // 10))
    else:         # soldier: narrow olive body
        cv2.rectangle(img, (cx - w // 2, cy - h // 2), (cx + w // 2, cy + h // 2),
                      (rng.randint(40, 70), rng.randint(60, 95), rng.randint(35, 60)), -1)
    return img

def synth_split(split, n, rng):
    os.makedirs(os.path.join(IMAGES, split), exist_ok=True)
    os.makedirs(os.path.join(LABELS, split), exist_ok=True)
    for i in range(n):
        img = rand_bg(IMG_SIZE, rng)
        horizon = IMG_SIZE * 0.42
        labels = []
        for _ in range(rng.randint(1, 5)):
            cls = rng.randint(0, 1)
            depth = rng.uniform(0.08, 0.9)                    # 0 = at horizon (small), 1 = close (large)
            cx = rng.randint(int(IMG_SIZE * 0.08), int(IMG_SIZE * 0.92))
            size_px = int((40 + 340 * depth ** 2))
            if cls == 0:
                w, h = size_px, max(4, int(size_px * 0.45))
            else:
                w, h = max(3, int(size_px * 0.22)), max(6, int(size_px * 0.85))
            cy = int(horizon + (IMG_SIZE - horizon) * depth)
            if cx - w // 2 < 2 or cx + w // 2 > IMG_SIZE - 2 or cy - h // 2 < 2:
                continue
            draw_target(img, cx, cy, w, h, cls, rng)
            labels.append(f'{cls} {cx / IMG_SIZE:.5f} {cy / IMG_SIZE:.5f} {w / IMG_SIZE:.5f} {h / IMG_SIZE:.5f}')
        cv2.imwrite(os.path.join(IMAGES, split, f'{split}_{i:05d}.jpg'), img)
        with open(os.path.join(LABELS, split, f'{split}_{i:05d}.txt'), 'w') as f:
            f.write('\n'.join(labels))

if not USE_ATTACHED:
    shutil.rmtree(DATASET_ROOT, ignore_errors=True)
    rng = random.Random(7)
    synth_split('train', 800, rng)
    synth_split('val', 200, rng)
    print('Synthetic dataset generated at', DATASET_ROOT)
else:
    DATASET_ROOT = attached
    print('Using attached dataset at', DATASET_ROOT)

In [ ]:
import yaml
data_yaml = os.path.join(WORK if not USE_ATTACHED else attached, 'masiez_rtdetr.yaml')
names = {0: 'tank', 1: 'soldier'}
if USE_ATTACHED and os.path.exists(os.path.join(attached, 'data.yaml')):
    data_yaml = os.path.join(attached, 'data.yaml')
else:
    with open(data_yaml, 'w') as f:
        yaml.safe_dump({'path': DATASET_ROOT, 'train': 'images/train', 'val': 'images/val',
                        'nc': len(CLASSES), 'names': names}, f)
print('Dataset config:', data_yaml)

## 3. Fine-tune RT-DETR

Start from the COCO-pretrained `rtdetr-l` so the base detector already understands "boxy ground vehicle" and "person"; a short fine-tune adapts it to the game's rendered look.

In [ ]:
from ultralytics import RTDETR

use_gpu = shutil.which('nvidia-smi') is not None
model = RTDETR('rtdetr-l.pt')
results = model.train(
    data=data_yaml,
    epochs=30,
    imgsz=IMG_SIZE,
    batch=8 if use_gpu else 2,
    device=0 if use_gpu else 'cpu',
    workers=2,
    project=os.path.join(WORK, 'runs'),
    name='rtdetr_masiez',
    patience=10,
)

In [ ]:
metrics = model.val(data=data_yaml, imgsz=IMG_SIZE, device=0 if use_gpu else 'cpu')
print('mAP50:', metrics.box.map50, '| mAP50-95:', metrics.box.map)
print('Per-class mAP50:', dict(zip(CLASSES, [float(v) for v in metrics.box.maps[:len(CLASSES)]])))

## 4. Export to ONNX for the browser

A **static 640x640** graph with simplified ops is what ONNX Runtime Web consumes fastest. `simplify=True` keeps the `1x300x6` output contract.

In [ ]:
onnx_path = model.export(format='onnx', imgsz=IMG_SIZE, simplify=True, dynamic=False, half=False)
onnx_path = str(onnx_path)

exported = os.path.join(WORK, 'rt-detr.onnx')
shutil.copy(onnx_path, exported)
with open(os.path.join(WORK, 'rt-detr.labels.txt'), 'w') as f:
    f.write('\n'.join(CLASSES) + '\n')
print('Model :', exported)
print('Labels:', os.path.join(WORK, 'rt-detr.labels.txt'))

## 5. Verify the exported graph matches the browser loader

`game/src/ai/RtDetrDetector.ts` expects one float input `[1,3,640,640]` in 0..1 and one output `[1,300,6]` (x1, y1, x2, y2, conf, class).

In [ ]:
import onnxruntime as ort

sess = ort.InferenceSession(exported, providers=['CPUExecutionProvider'])
inp = sess.get_inputs()[0]
out = sess.get_outputs()[0]
print('input :', inp.name, inp.shape, inp.type)
print('output:', out.name, out.shape, out.type)

dummy = np.random.rand(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
res = sess.run([out.name], {inp.name: dummy})
arr = np.asarray(res[0])
assert arr.shape[-1] == 6 and arr.ndim == 3, f'unexpected output shape {arr.shape}'
print('Inference OK — shape', arr.shape)

## 6. Ship it to the browser

1. Download `rt-detr.onnx` and `rt-detr.labels.txt` from this notebook's output panel.
2. Put both files in `game/public/models/` in the Masiez repo (they are git-ignored).
3. Start the game, open the User Menu (`~`) and set **IN-BROWSER AI** to DETECT or FOLLOW — the HUD chip should read `RT-DETR ONNX`.

Re-training note: if you change classes, update `CLASSES` here and `rt-detr.labels.txt` together — the label file's line order *is* the class-id mapping.